# Compare OSM vs. LiDAR Building Heights
This notebook visualizes and compares the building footprints with heights estimated from OpenStreetMap (levels * 3m) against the true physics-based heights extracted via LiDAR raster models.

In [ ]:
import geopandas as gpd
import pandas as pd
import osmnx as ox
import matplotlib.pyplot as plt
import numpy as np
from rasterstats import zonal_stats

# Suppress osmnx warnings
ox.settings.timeout = 1000

print("Loading reference streets to obtain the CRS...")
noise_streets = gpd.read_file("../layers/BCN_noise_streets.gpkg")
city_crs = noise_streets.crs
print("Active CRS:", city_crs)

## 1. Fetch OSM Building Geometries & Estimate Heights
We fetch the geometries and calculate `osm_estimated_height` using the exact same methodology as before.

In [ ]:
place_name = "Barcelona, Spain"
tags = {'building': True}

print("Downloading building footprints from OpenStreetMap...")
try:
    buildings = ox.features_from_place(place_name, tags)
except AttributeError:
    buildings = ox.geometries_from_place(place_name, tags)

buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
buildings = buildings.to_crs(city_crs)

# Process OSM Heights
def clean_height_col(series):
    if series.dtype == 'O':
        return pd.to_numeric(series.str.replace(r'[^\d.]', '', regex=True), errors='coerce')
    return pd.to_numeric(series, errors='coerce')

estimated_heights = pd.Series(index=buildings.index, dtype=float)

if 'height' in buildings.columns:
    estimated_heights = clean_height_col(buildings['height'])

if 'building:levels' in buildings.columns:
    levels = clean_height_col(buildings['building:levels'])
    height_from_levels = levels * 3.0
    estimated_heights = estimated_heights.fillna(height_from_levels)

median_height = estimated_heights.median()
if np.isnan(median_height):
    median_height = 9.0  
    
buildings['osm_estimated_height'] = estimated_heights.fillna(median_height)
print("Assigned OSM estimated heights.")

## 2. Extract Remote-Sensed LiDAR Heights
We evaluate the `zonal_stats` on the same building footprint geometries, using the mosaiced DSM for max roof elevation and DTM for mean terrain elevation.

In [ ]:
# Paths to our rasters
mosaic_dsm_path = "../data/processed/merged_lidar.tif"
dtm_path = "../layers/Digital_terrain_model/elevacions-terreny-lidar-Catalunya-2m-2008-2011tif1777940893149.tif"

print("Extracting max elevated top surfaces from DSM...")
stats_dsm = zonal_stats(buildings, mosaic_dsm_path, stats="max", all_touched=True, nodata=-9999.0)

print("Extracting mean ground terrain from DTM...")
stats_dtm = zonal_stats(buildings, dtm_path, stats="mean", all_touched=True, nodata=-9999.0)

df_dsm = pd.DataFrame(stats_dsm)
df_dtm = pd.DataFrame(stats_dtm)

# Fix index misalignment by passing .values directly, just as diagnosed previously
buildings['lidar_true_height'] = df_dsm['max'].values - df_dtm['mean'].values

# Clean out any NoData negatives
buildings['lidar_true_height'] = buildings['lidar_true_height'].clip(lower=0).fillna(0)
print("Assigned LiDAR true heights.")

# Calculate spatial differences
buildings['height_difference'] = buildings['lidar_true_height'] - buildings['osm_estimated_height']

## 3. Comparing Footprint Visualizations

In [ ]:
# Set a shared upper bound for color normalization so both maps are visually comparable
vmax = np.percentile(buildings['lidar_true_height'].dropna(), 95)
vmin = 0

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Map 1: OSM Heights
buildings.plot(
    column='osm_estimated_height', 
    cmap='viridis', 
    legend=True, 
    vmin=vmin, vmax=vmax,
    ax=axes[0]
)
axes[0].set_title("OSM Estimated Heights (m)")
axes[0].set_axis_off()

# Map 2: LiDAR Heights
buildings.plot(
    column='lidar_true_height', 
    cmap='viridis', 
    legend=True, 
    vmin=vmin, vmax=vmax,
    ax=axes[1]
)
axes[1].set_title("LiDAR True Heights (m)")
axes[1].set_axis_off()

# Map 3: Difference Map
diff_vmax = np.percentile(buildings['height_difference'].abs().dropna(), 95)
buildings.plot(
    column='height_difference', 
    cmap='RdBu', 
    legend=True,
    vmin=-diff_vmax, vmax=diff_vmax,
    ax=axes[2]
)
axes[2].set_title("Difference (LiDAR - OSM) (m)")
axes[2].set_axis_off()

plt.tight_layout()
plt.show()

## 4. Statistical Analysis
Let's see if the two metrics correlate well, and what the distribution of the errors looks like.

In [ ]:
from scipy.stats import pearsonr

# Filter out empty zero measurements or perfect matching defaults to focus on actual variation
valid_data = buildings[(buildings['osm_estimated_height'] > 0) & (buildings['lidar_true_height'] > 0)]

correlation, p_val = pearsonr(valid_data['osm_estimated_height'], valid_data['lidar_true_height'])

print(f"Overall Pearson correlation between OSM and LiDAR: {correlation:.3f}")
print(f"Average absolute difference (MAE): {valid_data['height_difference'].abs().mean():.2f} meters")

# Plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Scatter Plot
ax1.scatter(valid_data['osm_estimated_height'], valid_data['lidar_true_height'], alpha=0.1, color='blue', s=2)
ax1.plot([0, 100], [0, 100], color='red', linestyle='--') # Identity line
ax1.set_xlim(0, 75)
ax1.set_ylim(0, 75)
ax1.set_xlabel("OSM Estimated Height (m)")
ax1.set_ylabel("LiDAR True Height (m)")
ax1.set_title("OSM vs LiDAR Scatter")

# 2. Difference Histogram
ax2.hist(valid_data['height_difference'], bins=100, range=(-25, 25), color='purple', edgecolor='black')
ax2.set_xlabel("Difference: LiDAR - OSM (meters)")
ax2.set_ylabel("Number of Buildings")
ax2.set_title("Distribution of Discrepancies")

plt.show()